<a href="https://colab.research.google.com/github/trungkien1502/project_sna_group14/blob/main/03_E1_Full_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ==========================================
# E1 FULL TRAINING - CELL 1
# CONFIGURATION & ENVIRONMENT
# ==========================================

import sys
import torch
from pathlib import Path
from google.colab import drive

# ===== THÀNH VIÊN CHỈ CẦN CHỈNH 2 DÒNG =====

MODEL = "lightgcn_plus"       # lightgcn / lightgcn_plus
SEED = 2023              # 2023 / 2024 / 2025

# ===== FIXED CONFIG =====

DATASET = "amazon"
VALID_MODELS = ["lightgcn", "lightgcn_plus"]
VALID_SEEDS = [2023, 2024, 2025]

assert MODEL in VALID_MODELS, "Invalid MODEL"
assert SEED in VALID_SEEDS, "Invalid SEED"

# ===== GPU CHECK =====

print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Please enable GPU in Colab Runtime settings."
)

print("GPU:", torch.cuda.get_device_name(0))

# ===== GOOGLE DRIVE =====

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/Data_SNA_Group14"
)

DATA_DIR = ROOT / "data/raw/amazon"
REPO = Path("/content/RLMRec")

OUTPUT_DIR = (
    ROOT / "experiments/E1" /
    MODEL / f"seed_{SEED}"
)

assert DATA_DIR.is_dir(), "Dataset folder not found"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== SUMMARY =====

print("\n=== E1 CONFIGURATION ===")
print("Model:", MODEL)
print("Dataset:", DATASET)
print("Seed:", SEED)
print("Dataset path:", DATA_DIR)
print("Output path:", OUTPUT_DIR)

print("\nCELL 1: SUCCESS")


=== ENVIRONMENT ===
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== E1 CONFIGURATION ===
Model: lightgcn_plus
Dataset: amazon
Seed: 2023
Dataset path: /content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon
Output path: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023

CELL 1: SUCCESS


In [2]:

# ==========================================
# E1 FULL TRAINING - CELL 2
# REPOSITORY & DATASET SETUP
# ==========================================

import subprocess
from pathlib import Path

# Phiên bản đã smoke test thành công
RLMREC_COMMIT = "22413752246de3dee8ab0d509f7f7a8889080f95"
RLMREC_URL = "https://github.com/HKUDS/RLMRec.git"

# 1. Clone repository nếu chưa có
if not REPO.exists():
    subprocess.run(
        ["git", "clone", RLMREC_URL, str(REPO)],
        check=True
    )

# 2. Kiểm tra source chính thức
origin = subprocess.check_output(
    ["git", "-C", str(REPO), "remote", "get-url", "origin"],
    text=True
).strip()

assert origin == RLMREC_URL, (
    f"Repository không đúng nguồn RLMRec: {origin}"
)

# 3. Khóa phiên bản code
current_commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

if current_commit != RLMREC_COMMIT:
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        text=True
    ).strip()

    assert not status, (
        "Repository có thay đổi chưa lưu. "
        "Không tự động chuyển phiên bản."
    )

    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin"],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(REPO),
         "checkout", "--detach", RLMREC_COMMIT],
        check=True
    )

# 4. Kiểm tra 7 file dataset
required_files = [
    "trn_mat.pkl", "val_mat.pkl", "tst_mat.pkl",
    "usr_prf.pkl", "itm_prf.pkl",
    "usr_emb_np.pkl", "itm_emb_np.pkl"
]

for filename in required_files:
    assert (DATA_DIR / filename).is_file(), (
        f"Missing dataset file: {filename}"
    )

# 5. Kết nối dataset vào RLMRec
target = REPO / "data" / DATASET

if not target.exists() and not target.is_symlink():
    target.symlink_to(
        DATA_DIR,
        target_is_directory=True
    )

assert target.resolve() == DATA_DIR.resolve(), (
    "Dataset link không đúng"
)

# 6. Kiểm tra model configuration
model_config = (
    REPO / "encoder/config/modelconf"
    / f"{MODEL}.yml"
)

assert model_config.is_file(), (
    f"Missing model config: {model_config}"
)

assert (REPO / "encoder/train_encoder.py").is_file()

print("\n=== REPOSITORY SETUP ===")
print("Commit:", RLMREC_COMMIT)
print("Model:", MODEL)
print("Config:", model_config.name)
print("Dataset:", target.resolve())
print("Dataset files:", len(required_files))

print("\nCELL 2: SUCCESS")



=== REPOSITORY SETUP ===
Commit: 22413752246de3dee8ab0d509f7f7a8889080f95
Model: lightgcn_plus
Config: lightgcn_plus.yml
Dataset: /content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon
Dataset files: 7

CELL 2: SUCCESS


In [3]:

# ==========================================
# E1 FULL TRAINING - CELL 3
# TRAINING CONFIG & OUTPUT SETUP
# ==========================================

import json
import yaml
import sys
import torch
from pathlib import Path

# 1. Đọc cấu hình gốc của tác giả
CONFIG_PATH = (
    REPO / "encoder/config/modelconf" / f"{MODEL}.yml"
)

with open(CONFIG_PATH, "r") as f:
    original_config = yaml.safe_load(f)

assert original_config["model"]["name"] == MODEL

# 2. Giữ nguyên hyperparameters gốc
# Chỉ bật lưu checkpoint cho cả hai mô hình
training_config = {
    **original_config,
    "train": {
        **original_config["train"],
        "seed": SEED,
        "save_model": True
    }
}

# 3. Chuẩn bị thư mục lưu kết quả
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "logs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 4. Kết nối thư mục output của source gốc với Drive
def link_output(source, destination):
    source.parent.mkdir(parents=True, exist_ok=True)
    destination.mkdir(parents=True, exist_ok=True)

    if source.is_symlink():
        if source.resolve() != destination.resolve():
            source.unlink()
            source.symlink_to(destination, target_is_directory=True)
    elif source.exists():
        raise RuntimeError(
            f"{source} already exists and is not a symlink. "
            "Please inspect it before continuing."
        )
    else:
        source.symlink_to(destination, target_is_directory=True)

link_output(
    REPO / "encoder/checkpoint" / MODEL,
    CHECKPOINT_DIR
)

link_output(
    REPO / "encoder/log" / MODEL,
    LOG_DIR
)

# 5. Ghi lại thông tin thí nghiệm
run_info = {
    "experiment": "E1_Reproduction",
    "model": MODEL,
    "dataset": DATASET,
    "seed": SEED,
    "rlmrec_commit": RLMREC_COMMIT,
    "python_version": sys.version.split()[0],
    "pytorch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "training_config": training_config,
    "status": "prepared"
}

config_file = OUTPUT_DIR / "run_config.json"

config_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8"
)

# 6. Kiểm tra
print("=== TRAINING CONFIGURATION ===")
print("Model:", MODEL)
print("Seed:", SEED)
print("Max epochs:", training_config["train"]["epoch"])
print("Validation step:", training_config["train"]["test_step"])
print("Patience:", training_config["train"]["patience"])
print("Save checkpoint:", training_config["train"]["save_model"])

print("\n=== OUTPUT PATHS ===")
print("Checkpoints:", CHECKPOINT_DIR)
print("Logs:", LOG_DIR)
print("Config:", config_file)

print("\nCELL 3: SUCCESS")


=== TRAINING CONFIGURATION ===
Model: lightgcn_plus
Seed: 2023
Max epochs: 3000
Validation step: 3
Patience: 5
Save checkpoint: True

=== OUTPUT PATHS ===
Checkpoints: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/checkpoints
Logs: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/logs
Config: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/run_config.json

CELL 3: SUCCESS


In [4]:

# ==========================================
# E1 FULL TRAINING - CELL 4
# TRAIN + EARLY STOPPING + BEST CHECKPOINT
# ==========================================

import os
import sys
import json
import subprocess
import traceback
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

# 1. Kiểm tra cấu hình từ Cell 3
config_file = OUTPUT_DIR / "run_config.json"
assert config_file.is_file(), "Run Cell 3 first!"

run_info = json.loads(config_file.read_text())

assert run_info["model"] == MODEL
assert run_info["seed"] == SEED

checkpoint_file = (
    CHECKPOINT_DIR /
    f"{MODEL}-{DATASET}-{SEED}.pth"
)

metrics_file = OUTPUT_DIR / "metrics.json"
history_file = OUTPUT_DIR / "validation_history.json"
console_file = OUTPUT_DIR / "training_console.log"

# Tránh vô tình ghi đè một lần chạy hoàn chỉnh
assert not metrics_file.exists(), (
    "This run already has metrics.json. "
    "Do not overwrite an existing experiment."
)

assert not checkpoint_file.exists(), (
    "Checkpoint already exists for this model/seed."
)

assert (
    REPO / "encoder/checkpoint" / MODEL
).resolve() == CHECKPOINT_DIR.resolve()

# 2. Chuẩn bị imports từ repository gốc
os.chdir(REPO)

# Import config phải xảy ra sau khi đặt model arguments
assert "config.configurator" not in sys.modules, (
    "RLMRec config already imported. "
    "Restart runtime and run Cells 1-4 in order."
)

original_argv = sys.argv[:]
sys.argv = [
    "train_encoder.py",
    "--model", MODEL,
    "--dataset", DATASET,
    "--cuda", "0",
    "--seed", str(SEED)
]

try:
    sys.path.insert(0, str(REPO / "encoder"))

    from config.configurator import configs
    from trainer.trainer import init_seed
    from data_utils.build_data_handler import build_data_handler
    from models.bulid_model import build_model
    from trainer.logger import Logger
    from trainer.build_trainer import build_trainer
finally:
    sys.argv = original_argv

# 3. Áp dụng cấu hình đã chuẩn bị ở Cell 3
configs["train"].update(
    run_info["training_config"]["train"]
)

configs["train"]["seed"] = SEED
configs["train"]["save_model"] = True

assert configs["model"]["name"] == MODEL
assert configs["test"]["k"][-1] == 20

# 4. Khởi tạo pipeline
init_seed()

handler = build_data_handler()
handler.load_data()

model = build_model(handler).to(configs["device"])

logger = Logger()
trainer = build_trainer(handler, logger)

# 5. Theo dõi validation & best epoch
history = []
best = {"epoch": None, "recall20": -1.0}
captured_test = {}
epoch_counter = {"completed": 0}

original_evaluate = trainer.evaluate
original_test = trainer.test
original_train_epoch = trainer.train_epoch

K_VALUES = configs["test"]["k"]

def format_metrics(raw):
    return {
        f"{metric}@{k}": float(value)
        for metric, values in raw.items()
        for k, value in zip(K_VALUES, values)
    }

def tracked_evaluate(current_model, epoch_idx=None):
    result = original_evaluate(current_model, epoch_idx)

    if epoch_idx is not None:
        values = format_metrics(result)
        recall20 = values["recall@20"]

        history.append({
            "epoch_index": int(epoch_idx),
            **values
        })

        if recall20 > best["recall20"]:
            best["epoch"] = int(epoch_idx)
            best["recall20"] = recall20

        history_file.write_text(
            json.dumps(history, indent=2),
            encoding="utf-8"
        )

    return result

def tracked_test(current_model):
    result = original_test(current_model)
    captured_test.update(format_metrics(result))
    return result

def tracked_train_epoch(current_model, epoch_idx):
    original_train_epoch(current_model, epoch_idx)
    epoch_counter["completed"] += 1

trainer.evaluate = tracked_evaluate
trainer.test = tracked_test
trainer.train_epoch = tracked_train_epoch

# 6. Ghi console ra cả notebook và Drive
class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()

run_info["status"] = "running"
config_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8"
)

print("=== E1 FULL TRAINING ===")
print("Model:", MODEL)
print("Seed:", SEED)
print("Max epochs:", configs["train"]["epoch"])
print("Patience:", configs["train"]["patience"])
print("Checkpoint:", checkpoint_file)
print("Starting training...\n")

# 7. Full Training bằng trainer gốc
try:
    with open(console_file, "a", encoding="utf-8") as log:
        with redirect_stdout(Tee(sys.stdout, log)):
            with redirect_stderr(log):
                trainer.train(model)
except Exception:
    run_info["status"] = "failed"
    config_file.write_text(
        json.dumps(run_info, indent=2),
        encoding="utf-8"
    )
    print("TRAINING FAILED")
    print(traceback.format_exc())
    raise

# 8. Kiểm tra kết quả cuối cùng
assert checkpoint_file.is_file(), (
    "Training ended but checkpoint was not saved!"
)

assert checkpoint_file.stat().st_size > 0
assert captured_test, "Missing final test metrics!"
assert best["epoch"] is not None

# 9. Lưu metrics
result = {
    "experiment": "E1_Reproduction",
    "model": MODEL,
    "dataset": DATASET,
    "seed": SEED,
    "rlmrec_commit": RLMREC_COMMIT,
    "epochs_completed": epoch_counter["completed"],
    "best_epoch_index": best["epoch"],
    "best_epoch_number": best["epoch"] + 1,
    "best_validation_recall20": best["recall20"],
    "test_metrics": captured_test,
    "checkpoint": str(checkpoint_file),
    "status": "completed"
}

metrics_file.write_text(
    json.dumps(result, indent=2),
    encoding="utf-8"
)

run_info["status"] = "completed"
run_info["epochs_completed"] = epoch_counter["completed"]
run_info["best_epoch_index"] = best["epoch"]

config_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8"
)

print("\n=== E1 TRAINING COMPLETED ===")
print("Epochs completed:", result["epochs_completed"])
print("Best epoch:", result["best_epoch_number"])
print("Best validation Recall@20:",
      result["best_validation_recall20"])

print("\n=== FINAL TEST METRICS ===")
for metric, value in captured_test.items():
    print(f"{metric}: {value:.6f}")

print("\n=== SAVED FILES ===")
print("Metrics:", metrics_file)
print("Checkpoint:", checkpoint_file)
print("Validation history:", history_file)
print("Console log:", console_file)

print("\nCELL 4: SUCCESS")


/content/RLMRec/encoder/data_utils/data_handler_general_cf.py:86: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  return t.sparse.FloatTensor(idxs, vals, shape).to(configs['device'])
INFO:train_logger:{'optimizer': {'name': 'adam', 'lr': 0.001, 'weight_decay': 0}, 'train': {'epoch': 3000, 'batch_size': 4096, 'save_model': True, 'loss': 'pairwise', 'test_step': 3, 'reproducible': True, 'seed': 2023, 'patience': 5}, 'test': {'metrics': ['recall', 'ndcg'], 'k': [5, 10, 20], 'batch_size': 1024}, 'data': {'type': 'general_cf', 'name': 'amazon', 'user_num': 11000, 'item_num': 9332}, 'model': {'name': 'lightgcn_plus', 'keep_rate': 0.8, 'embedding_size': 32, 'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2, 'amazon': {'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, '

{'optimizer': {'name': 'adam', 'lr': 0.001, 'weight_decay': 0}, 'train': {'epoch': 3000, 'batch_size': 4096, 'save_model': True, 'loss': 'pairwise', 'test_step': 3, 'reproducible': True, 'seed': 2023, 'patience': 5}, 'test': {'metrics': ['recall', 'ndcg'], 'k': [5, 10, 20], 'batch_size': 1024}, 'data': {'type': 'general_cf', 'name': 'amazon', 'user_num': 11000, 'item_num': 9332}, 'model': {'name': 'lightgcn_plus', 'keep_rate': 0.8, 'embedding_size': 32, 'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2, 'amazon': {'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2}, 'yelp': {'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2}, 'steam': {'layer_num': 3, 'reg_weight': 1e-08, 'kd_weight': 0.001, 'kd_temperature': 0.1}}}
=== E1 FULL TRAINING ===
Model: lightgcn_plus
Seed: 2023
Max epochs: 3000
Patience: 5
Checkpoint: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/checkpoints/li

INFO:train_logger:Epoch   0 Validation set [recall@5: 0.0310 recall@10: 0.0511 recall@20: 0.0818 ] [ndcg@5: 0.0302 ndcg@10: 0.0374 ndcg@20: 0.0475 ] 


Epoch   0 Validation set [recall@5: 0.0310 recall@10: 0.0511 recall@20: 0.0818 ] [ndcg@5: 0.0302 ndcg@10: 0.0374 ndcg@20: 0.0475 ] 


INFO:train_logger:Epoch   3 Validation set [recall@5: 0.0299 recall@10: 0.0495 recall@20: 0.0772 ] [ndcg@5: 0.0291 ndcg@10: 0.0362 ndcg@20: 0.0454 ] 


...

Epoch 153 Validation set [recall@5: 0.0626 recall@10: 0.0996 recall@20: 0.1492 ] [ndcg@5: 0.0617 ndcg@10: 0.0745 ndcg@20: 0.0908 ] 


INFO:train_logger:[recall@5: 0.0621 recall@10: 0.0987 recall@20: 0.1509 ] [ndcg@5: 0.0614 ndcg@10: 0.0742 ndcg@20: 0.0912 ] 


[recall@5: 0.0621 recall@10: 0.0987 recall@20: 0.1509 ] [ndcg@5: 0.0614 ndcg@10: 0.0742 ndcg@20: 0.0912 ] 


INFO:train_logger:[recall@5: 0.0616 recall@10: 0.0984 recall@20: 0.1480 ] [ndcg@5: 0.0617 ndcg@10: 0.0747 ndcg@20: 0.0911 ] 
INFO:train_logger:Save model parameters to ./encoder/checkpoint/lightgcn_plus/lightgcn_plus-amazon-2023.pth
INFO:train_logger:Best Epoch 138. Final test result: {'recall': array([0.06156244, 0.09835712, 0.14797552]), 'ndcg': array([0.06170561, 0.0747389 , 0.0910959 ])}.


[recall@5: 0.0616 recall@10: 0.0984 recall@20: 0.1480 ] [ndcg@5: 0.0617 ndcg@10: 0.0747 ndcg@20: 0.0911 ] 
Save model parameters to ./encoder/checkpoint/lightgcn_plus/lightgcn_plus-amazon-2023.pth
Best Epoch 138. Final test result: {'recall': array([0.06156244, 0.09835712, 0.14797552]), 'ndcg': array([0.06170561, 0.0747389 , 0.0910959 ])}.

=== E1 TRAINING COMPLETED ===
Epochs completed: 154
Best epoch: 139
Best validation Recall@20: 0.15085578005692052

=== FINAL TEST METRICS ===
recall@5: 0.061562
recall@10: 0.098357
recall@20: 0.147976
ndcg@5: 0.061706
ndcg@10: 0.074739
ndcg@20: 0.091096

=== SAVED FILES ===
Metrics: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/metrics.json
Checkpoint: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/checkpoints/lightgcn_plus-amazon-2023.pth
Validation history: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/validation_history.json
Console log: /content/dr

In [5]:

# ==========================================
# E1 FULL TRAINING - CELL 5
# VERIFY TRAINING RESULTS
# ==========================================

import json
import math
import torch
from pathlib import Path
from datetime import datetime, timezone

# 1. Xác định các file cần kiểm tra
config_file = OUTPUT_DIR / "run_config.json"
metrics_file = OUTPUT_DIR / "metrics.json"
history_file = OUTPUT_DIR / "validation_history.json"
console_file = OUTPUT_DIR / "training_console.log"

checkpoint_file = (
    CHECKPOINT_DIR / f"{MODEL}-{DATASET}-{SEED}.pth"
)

required_files = {
    "Run configuration": config_file,
    "Test metrics": metrics_file,
    "Validation history": history_file,
    "Training log": console_file,
    "Best checkpoint": checkpoint_file
}

print("=== E1 OUTPUT VERIFICATION ===")

# 2. Kiểm tra file tồn tại và không rỗng
for label, path in required_files.items():
    assert path.is_file(), f"Missing {label}: {path}"
    assert path.stat().st_size > 0, f"Empty {label}"

    print(f"[PASS] {label}")

# 3. Kiểm tra metadata
config = json.loads(config_file.read_text())
metrics = json.loads(metrics_file.read_text())
history = json.loads(history_file.read_text())

assert config["status"] == "completed"
assert metrics["status"] == "completed"
assert metrics["model"] == MODEL
assert metrics["seed"] == SEED
assert metrics["rlmrec_commit"] == RLMREC_COMMIT

assert metrics["epochs_completed"] >= 1
assert len(history) >= 1

print("[PASS] Run metadata")

# 4. Kiểm tra Recall / NDCG
test_metrics = metrics["test_metrics"]

required_metrics = [
    "recall@10",
    "recall@20",
    "ndcg@10",
    "ndcg@20"
]

for name in required_metrics:
    assert name in test_metrics, f"Missing metric: {name}"

    value = float(test_metrics[name])

    assert math.isfinite(value), f"Invalid metric: {name}"
    assert 0 <= value <= 1, f"Out-of-range metric: {name}"

assert test_metrics["recall@10"] <= test_metrics["recall@20"]

print("[PASS] Test metrics")

# 5. Kiểm tra validation history
best_recall = max(
    float(record["recall@20"])
    for record in history
)

assert math.isclose(
    best_recall,
    float(metrics["best_validation_recall20"]),
    rel_tol=1e-5,
    abs_tol=1e-7
)

assert any(
    record["epoch_index"] == metrics["best_epoch_index"]
    for record in history
)

print("[PASS] Best validation checkpoint selection")

# 6. Thử đọc checkpoint từ Google Drive
state_dict = torch.load(
    checkpoint_file,
    map_location="cpu",
    weights_only=True
)

assert isinstance(state_dict, dict)
assert len(state_dict) > 0

assert all(
    isinstance(value, torch.Tensor)
    for value in state_dict.values()
)

print("[PASS] Checkpoint can be loaded")

# 7. Kiểm tra log gốc
source_logs = list(LOG_DIR.glob("*.log"))

assert len(source_logs) >= 1, (
    "Missing original RLMRec training log"
)

print("[PASS] Original training logs")

# 8. Lưu báo cáo nghiệm thu
verification = {
    "experiment": "E1_Reproduction",
    "model": MODEL,
    "seed": SEED,
    "status": "passed",
    "verified_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "epochs_completed": metrics["epochs_completed"],
    "best_epoch": metrics["best_epoch_number"],
    "best_validation_recall20": best_recall,
    "test_metrics": {
        name: float(test_metrics[name])
        for name in required_metrics
    },
    "checkpoint_readable": True,
    "checkpoint_parameter_tensors": len(state_dict)
}

verification_file = OUTPUT_DIR / "verification.json"

verification_file.write_text(
    json.dumps(verification, indent=2),
    encoding="utf-8"
)

# 9. Kết quả cuối
print("\n=== FINAL RESULTS ===")
print("Model:", MODEL)
print("Seed:", SEED)
print("Epochs:", metrics["epochs_completed"])
print("Best epoch:", metrics["best_epoch_number"])

for name in required_metrics:
    print(f"{name}: {test_metrics[name]:.6f}")

print("\nVerification report:", verification_file)
print("\nCELL 5: SUCCESS")


=== E1 OUTPUT VERIFICATION ===
[PASS] Run configuration
[PASS] Test metrics
[PASS] Validation history
[PASS] Training log
[PASS] Best checkpoint
[PASS] Run metadata
[PASS] Test metrics
[PASS] Best validation checkpoint selection
[PASS] Checkpoint can be loaded
[PASS] Original training logs

=== FINAL RESULTS ===
Model: lightgcn_plus
Seed: 2023
Epochs: 154
Best epoch: 139
recall@10: 0.098357
recall@20: 0.147976
ndcg@10: 0.074739
ndcg@20: 0.091096

Verification report: /content/drive/MyDrive/Data_SNA_Group14/experiments/E1/lightgcn_plus/seed_2023/verification.json

CELL 5: SUCCESS
